# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a comprehensive guide for loading and exploring the [FAIR^2 dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List record sets and their fields by @id
record_sets = metadata.record_sets  # This is a list of mlcroissant.RecordSet
for rs in record_sets:
    print(f"RecordSet: {rs.name} (@id: {rs.id})")
    for field in rs.fields:
        print(f"  Field: {field.name} (@id: {field.id}, type: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

> **Tip:** All accesses use `@id`, not `name`.

In [ ]:
# Get all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Pick the first record set as a main table for demonstration
main_record_set_id = record_set_ids[0]
print(f"Loaded DataFrame columns in {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# For demonstration, select a numeric field from the fields listed earlier.
# For mass spectrometry data, 'MW [kDa]' (molecular weight), 'TotalPeptides', or 'Coverage [percent]' are likely candidates.

# Let's programmatically select the first numeric field from the record set fields
main_record_set = [rs for rs in metadata.record_sets if rs.id == main_record_set_id][0]
numeric_field = None
for field in main_record_set.fields:
    # Typical numeric data types are 'Number', 'Float', 'Integer'
    if field.data_type.lower() in ('float', 'number', 'integer', 'double'):
        numeric_field = field.id
        break

if not numeric_field:
    print('No numeric field found.')
else:
    print(f"Using numeric field for EDA: {numeric_field}")
    df = dataframes[main_record_set_id]

    # Clean column names if necessary (some fields may have characters like spaces or units)
    if numeric_field not in df.columns:
        # Try to match by lower-case and ignore spaces/units
        candidates = [col for col in df.columns if numeric_field.lower() in col.lower()]
        if candidates:
            numeric_field = candidates[0]

    # Convert to numeric type if not already (may have been loaded as string)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].median() # Use median value as threshold

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by another categorical field if available
    group_field = None
    for field in main_record_set.fields:
        # Exclude the numeric field itself and try for string/categorical types
        if field.id != numeric_field and field.data_type.lower() in ('string', 'text'):
            group_field = field.id
            break

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For demonstration, we'll plot the distribution of the selected numeric field and the normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    if norm_col in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[norm_col].dropna(), kde=True, color='orange', bins=30)
        plt.xlabel(norm_col)
        plt.title(f"Distribution of Normalized {numeric_field} (filtered)")
        plt.show()


## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using the `mlcroissant` library, referencing entity `@id`s throughout for reproducibility. We identified available record sets and fields, filtered and normalized a numeric value, and visualized feature distribution. For full metadata details and further exploration, use the rich introspection tools provided by `mlcroissant`.